# Create Mapbox GeoJSON from Business Panel and License Data

Input:
- Longitudinal business panel dataset (`joined_dataset.csv`)
- License dataset (`licenses.csv`)

Processes:
- Cleans and standardizes key fields
- Merges business records with exact corresponding license coordinates
- Aggregates business activity metrics
- Determines whether each business is active as of March 2026 based on last observed month in panel data

Output:
- `businesses.geojson` file saved to Google Drive containing point features for use in a Mapbox GL JS visualization

### 1. Mount Google Drive and import libraries

In [11]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import pandas as pd
import json

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


### 2. Define filepaths and load datasets

In [12]:
base = Path("/content/drive/MyDrive")

joined_path = base / "joined_dataset.csv"
licenses_path = base / "licenses.csv"
output_path = base / "businesses.geojson"

joined = pd.read_csv(joined_path)
licenses = pd.read_csv(licenses_path)

print("joined shape:", joined.shape)
print("licenses shape:", licenses.shape)

joined shape: (2240985, 236)
licenses shape: (32432, 31)


### 3. Clean merge keys and parse dates

In [13]:
joined["business_id"] = joined["business_id"].astype(str).str.strip()
licenses["Business Unique ID"] = licenses["Business Unique ID"].astype(str).str.strip()

joined["month"] = pd.to_datetime(joined["month"], errors="coerce")

### 4. Clean license dataset fields

In [14]:
string_cols = [
    "Business Name", "Contact Phone", "Building Number", "Street1", "Street2",
    "Street3", "Apt/Suite", "City", "State", "ZIP Code", "Borough"
]

for col in string_cols:
    if col in licenses.columns:
        licenses[col] = licenses[col].astype(str).str.strip()

licenses["Latitude"] = pd.to_numeric(licenses["Latitude"], errors="coerce")
licenses["Longitude"] = pd.to_numeric(licenses["Longitude"], errors="coerce")

### 5. Construct full business addresses

In [15]:
address_parts = ["Building Number", "Street1", "Street2", "Street3", "Apt/Suite"]
existing_address_parts = [c for c in address_parts if c in licenses.columns]

licenses["full_address"] = (
    licenses[existing_address_parts]
    .fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

licenses["full_address"] = (
    licenses["full_address"]
    + ", "
    + licenses["City"].fillna("").astype(str).str.strip()
    + " "
    + licenses["State"].fillna("").astype(str).str.strip()
    + " "
    + licenses["ZIP Code"].fillna("").astype(str).str.strip()
).str.replace(r"\s+", " ", regex=True).str.strip(" ,")

### 6. Extract one coordinate per business

In [16]:
license_geo = (
    licenses.dropna(subset=["Latitude", "Longitude"])
    .sort_values(["Business Unique ID"])
    .drop_duplicates(subset=["Business Unique ID"], keep="first")
    [[
        "Business Unique ID",
        "Business Name",
        "full_address",
        "Borough",
        "Contact Phone",
        "Latitude",
        "Longitude"
    ]]
    .rename(columns={
        "Business Unique ID": "business_id",
        "Business Name": "business_name",
        "full_address": "license_address",
        "Borough": "license_borough",
        "Contact Phone": "license_contact_phone",
        "Latitude": "latitude",
        "Longitude": "longitude"
    })
)

print("Businesses with usable coordinates:", license_geo["business_id"].nunique())

Businesses with usable coordinates: 18549


### 7. Build business-level summary (as of March 2026)

In [17]:
cutoff = pd.Timestamp("2026-03-01")

business_summary = (
    joined.sort_values(["business_id", "month"])
    .groupby("business_id", as_index=False)
    .agg(
        business_category_sum=("business_category_sum", "sum"),
        complaint_sum=("complaint_sum", "sum"),
        last_month=("month", "max"),
        last_open=("open", "last")
    )
)

business_summary["active"] = (business_summary["last_month"] >= cutoff).astype(int)

print("Active/inactive counts:")
print(business_summary["active"].value_counts())

Active/inactive counts:
active
1    12473
0     6069
Name: count, dtype: int64


### 8. Merge business summary with geographic data

In [18]:
businesses = business_summary.merge(
    license_geo,
    on="business_id",
    how="left"
)

print("Merged dataset shape:", businesses.shape)
print("Businesses with coordinates:", businesses[["latitude","longitude"]].dropna().shape[0])

Merged dataset shape: (18542, 12)
Businesses with coordinates: 18542


### 9. Convert dataset into GeoJSON features

In [19]:
businesses = businesses.where(pd.notna(businesses), None)

features = []

for _, row in businesses.iterrows():

    lat = row["latitude"]
    lng = row["longitude"]

    if lat is None or lng is None:
        continue

    features.append({
        "type": "Feature",
        "properties": {
            "business_id": row["business_id"],
            "business_name": row["business_name"],
            "address": row["license_address"],
            "borough": row["license_borough"],
            "contact_phone": row["license_contact_phone"],
            "business_category_sum": row["business_category_sum"],
            "complaint_sum": row["complaint_sum"],
            "active": int(row["active"]) if row["active"] is not None else None,
            "last_month": row["last_month"].strftime("%Y-%m-%d") if row["last_month"] is not None else None
        },
        "geometry": {
            "type": "Point",
            "coordinates": [float(lng), float(lat)]
        }
    })

### 10. Save GeoJSON into GDrive

In [20]:
geojson = {
    "type": "FeatureCollection",
    "features": features
}

with open(output_path, "w") as f:
    json.dump(geojson, f, indent=2, allow_nan=False)

print("Saved GeoJSON to:", output_path)
print("Number of features:", len(features))

Saved GeoJSON to: /content/drive/MyDrive/businesses.geojson
Number of features: 18542


### 11. Validate GeoJSON and perform sanity checks

In [21]:
with open(output_path, "r") as f:
    check = json.load(f)

print("GeoJSON type:", check["type"])
print("Number of features:", len(check["features"]))

coord_df = businesses[["latitude","longitude"]].dropna().drop_duplicates()
print("Unique coordinate pairs:", len(coord_df))

print("\nExample inactive businesses:")
display(
    businesses[businesses["active"] == 0][
        ["business_id","business_name","last_month","active"]
    ].head(20)
)

GeoJSON type: FeatureCollection
Number of features: 18542
Unique coordinate pairs: 15506

Example inactive businesses:


,business_id,business_name,last_month,active
4,BA-1010980-2022,10TH STREET FOOD CORP,2022-12-01,0
10,BA-1019920-2022,55 CORNER DELI INC,2023-12-01,0
17,BA-1035521-2022,ALL CITY FENCE INC,2025-02-01,0
19,BA-1038947-2022,"AG Wood Works, Inc.",2025-02-01,0
22,BA-1041032-2022,ANCOR GOLD INC,2025-07-01,0
23,BA-1041058-2022,ALRIC PARKING LLC,2025-03-01,0
24,BA-1041069-2022,3RD AVE VAP INC,2023-11-01,0
25,BA-1041094-2022,170 ELECTRONICS WORLD USA INC,2024-12-01,0
28,BA-1042503-2022,"AMERICAN FLAG DELI AND GROCERY, INC.",2025-12-01,0
30,BA-1046661-2022,BROADWAY NEWS STAND CORP.,2023-12-01,0
